In [1]:
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from dotenv import load_dotenv
import json
import os

from demo_db import get_db_url

load_dotenv()

ukrdc3_sessionmaker = sessionmaker(
    autocommit=False, autoflush=False, bind=create_engine(get_db_url())
)

ukrdc3 = ukrdc3_sessionmaker()


In [2]:
from ukrdc.database import Connection
from sqlalchemy.orm import sessionmaker

engine = Connection.get_engine_from_file(key="ukrdc_staging")


ukrdc3_sessionmaker = sessionmaker(
    autocommit=False, autoflush=False, bind=engine
)

ukrdc3 = ukrdc3_sessionmaker()


# Patient Demographic Statistics 

This is the simplest group of stats returned to the UI. It returns binned data directly calculated from fields in the Patient table of the UKRDC ([documentation here](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450145/Patient)). 


These stats are being calculated in this demo for unit "RJZ". Specifically it is a count of the total number of living patients, who have ever appeared in a UKRDC feed from that particular unit, with an entry into a given Patient field. 

The properties which get returned are:
- <strong> Ethnic group: </strong> the ethnic group from the patient table with the five ethnicity grouping used in the annual report and data portal.  
- <strong> Gender: </strong> the gender from the patient table interpreted as the person stated gender code in the NHS digital guidelines. 
- <strong> Age: </strong> The integer age in years (i.e taking account leap years and rounding down) calculated from the time of birth recorded in the patient table. 

The library returns the output as an API friendly pydantic object. The following  cell shows how to return raw stats output. 

In [3]:
from ukrdc_stats.calculators.demographics import DemographicStatsCalculator
import datetime as dt
from IPython.display import display
import json


# initialize the demographic stats calculator
calculator = DemographicStatsCalculator(ukrdc3, "RJZ")

# run function to extract stats
output = calculator.extract_stats()

# results are returned as pydantic object. To make these easier to 
formatted_output = json.dumps(
    json.loads(
        output.json()
    ), 
    indent=4
)

print(formatted_output)


debug 1
{
    "gender": {
        "metadata": {
            "title": "Gender Distribution",
            "summary": "Breakdown of patient gender identity codes",
            "description": "\n# Patient Gender\nGender identity recorded for each living patient registered with the renal unit.\n\n# Methodology\n- Patient records are matched to NHS stated gender using patient demographic information \n- Patients are optionally checked against NHS tracing to check for date of death\n- All living patients with patient records sent by a particular sending facility are aggregated based on gender\n\n\n## UKRDC Entities Used\n- [PatientRecord](https://renalregistry.atlassian.net/l/cp/KCZ6A2bX)\n- [Patient](https://renalregistry.atlassian.net/l/cp/0MXHtpTU)\n\n",
            "axis_titles": {
                "x": "Gender",
                "y": "No. of Patients"
            },
            "population_size": null,
            "coding_standard_x": null,
            "units_y": null
        },
        "d

The ukrdc frontend uses plotly to visualise the data. Plotly is cross platform and can be used to make plots and dashboards in python. The following snippit of code will make a set of visualisations based on the output of the DemographicCalculator. 

The figures are currently being rendered in png mode so they persist when uploaded to github. This can be disabled when run locally to allow more interactivity.  

In [5]:
import time 
print(time.time())
t1 = time.time()
calculator._extract_base_patient_cohort()

t2 = time.time()
calculator._extract_base_patient_cohort(include_tracing=True)

t3 = time.time()
print(t2-t1)
print(t3-t2)


1723554020.0988948


AttributeError: 'DataFrame' object has no attribute 'deathtime'

In [6]:
# visualise gender distribution with plotly
from cProfile import label
from importlib.metadata import metadata
import plotly.express as px

age_distribution = px.bar(
    x=output.age.data.x,
    y=output.age.data.y,
    title=output.age.metadata.title,
    labels={
        "x": output.age.metadata.axis_titles.x,
        "y": output.age.metadata.axis_titles.y,
    },
)
age_distribution.show()


gender_fig = px.pie(
    values=output.gender.data.y,
    names=output.gender.data.x,
    title=output.gender.metadata.title,
    hole=0.3,
)

gender_fig.show()

# plot bar charts of some other bits
ethnicity_fig = px.pie(
    values=output.ethnic_group.data.y,
    names=output.ethnic_group.data.x,
    title=output.ethnic_group.metadata.title,
    hole=0.3,
)
ethnicity_fig.show()
